In [1]:
!nvidia-smi

Mon May 11 18:18:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matrix_multiplication.cu

#include <iostream>
using namespace std;

// CUDA Kernel for Matrix Multiplication
__global__ void matrixMultiplication(
    int *A,
    int *B,
    int *C,
    int rowsA,
    int colsA,
    int colsB
)
{
    int row =
        blockIdx.y * blockDim.y
        + threadIdx.y;

    int col =
        blockIdx.x * blockDim.x
        + threadIdx.x;

    if(row < rowsA && col < colsB)
    {
        int sum = 0;

        for(int k = 0; k < colsA; k++)
        {
            sum +=
                A[row * colsA + k] *
                B[k * colsB + col];
        }

        C[row * colsB + col] = sum;
    }
}

int main()
{
    int rowsA, colsA, rowsB, colsB;

    cout << "Enter rows and columns of Matrix A: ";
    cin >> rowsA >> colsA;

    cout << "Enter rows and columns of Matrix B: ";
    cin >> rowsB >> colsB;

    // Check multiplication condition
    if(colsA != rowsB)
    {
        cout << "Matrix multiplication not possible!\n";
        return 0;
    }

    int sizeA = rowsA * colsA;
    int sizeB = rowsB * colsB;
    int sizeC = rowsA * colsB;

    // Host matrices
    int *h_A = new int[sizeA];
    int *h_B = new int[sizeB];
    int *h_C = new int[sizeC];

    // Input Matrix A
    cout << "Enter elements of Matrix A:\n";

    for(int i = 0; i < sizeA; i++)
    {
        cin >> h_A[i];
    }

    // Input Matrix B
    cout << "Enter elements of Matrix B:\n";

    for(int i = 0; i < sizeB; i++)
    {
        cin >> h_B[i];
    }

    // Device matrices
    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, sizeA * sizeof(int));
    cudaMalloc(&d_B, sizeB * sizeof(int));
    cudaMalloc(&d_C, sizeC * sizeof(int));

    // Copy data from Host to Device
    cudaMemcpy(
        d_A,
        h_A,
        sizeA * sizeof(int),
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_B,
        h_B,
        sizeB * sizeof(int),
        cudaMemcpyHostToDevice
    );

    // Define block and grid size
    dim3 threadsPerBlock(16, 16);

    dim3 blocksPerGrid(
        (colsB + 15) / 16,
        (rowsA + 15) / 16
    );

    // Launch Kernel
    matrixMultiplication<<<blocksPerGrid, threadsPerBlock>>>(
        d_A,
        d_B,
        d_C,
        rowsA,
        colsA,
        colsB
    );

    // Copy result back to Host
    cudaMemcpy(
        h_C,
        d_C,
        sizeC * sizeof(int),
        cudaMemcpyDeviceToHost
    );

    // Display Result Matrix
    cout << "\nResultant Matrix:\n";

    for(int i = 0; i < rowsA; i++)
    {
        for(int j = 0; j < colsB; j++)
        {
            cout << h_C[i * colsB + j] << " ";
        }

        cout << endl;
    }

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    return 0;
}

Writing matrix_multiplication.cu


In [3]:
!nvcc matrix_multiplication.cu -o matrix_multiplication

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./matrix_multiplication

Enter rows and columns of Matrix A: 3 3
Enter rows and columns of Matrix B: 3 3
Enter elements of Matrix A:
1 5 2
9 3 1
4 6 7
Enter elements of Matrix B:
3 6 8
5 7 2
1 2 4

Resultant Matrix:
30 45 26 
43 77 82 
49 80 72 
